<a href="https://colab.research.google.com/github/Tanishqnarayan/AIML/blob/main/AIML_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:



!pip -q install ultralytics

import os
import cv2
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from ultralytics import YOLO
from google.colab import files
from IPython.display import display, Video


# ============================================================
# 3. UPLOAD CCTV VIDEO
# ============================================================

print("Please upload your CCTV/video file.")

uploaded = files.upload()

video_file = list(uploaded.keys())[0]

print("\nUploaded video:")
print(video_file)


# ============================================================
# 4. LOAD YOLO MODEL
# ============================================================

print("\nLoading YOLOv8 model...")

model = YOLO("yolov8n.pt")

print("YOLO model loaded successfully!")


# ============================================================
# 5. OPEN VIDEO
# ============================================================

cap = cv2.VideoCapture(video_file)

if not cap.isOpened():
    raise Exception("Could not open the uploaded video.")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

duration = total_frames / fps if fps > 0 else 0

print("\n================ VIDEO INFORMATION ================")
print("Width:", width)
print("Height:", height)
print("FPS:", round(fps, 2))
print("Total Frames:", total_frames)
print("Duration:", round(duration, 2), "seconds")


# ============================================================
# 6. CREATE OUTPUT VIDEO
# ============================================================

output_video = "AIML_9_CCTV_Object_Detection.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_video,
    fourcc,
    fps,
    (width, height)
)


# ============================================================
# 7. OBJECT DETECTION
# ============================================================

print("\nStarting object detection...")
print("This may take some time depending on video length.")

detection_records = []

frame_number = 0

# Count detections
object_counter = Counter()

# Confidence threshold
CONFIDENCE_THRESHOLD = 0.35


while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # Run YOLO detection
    results = model(
        frame,
        conf=CONFIDENCE_THRESHOLD,
        verbose=False
    )

    result = results[0]

    # Get annotated frame
    annotated_frame = result.plot()

    # Process detections
    if result.boxes is not None:

        for box in result.boxes:

            class_id = int(box.cls[0])
            confidence = float(box.conf[0])

            class_name = model.names[class_id]

            # Bounding box coordinates
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            x1 = int(x1)
            y1 = int(y1)
            x2 = int(x2)
            y2 = int(y2)

            object_counter[class_name] += 1

            detection_records.append({
                "Frame": frame_number,
                "Object": class_name,
                "Confidence": confidence,
                "X1": x1,
                "Y1": y1,
                "X2": x2,
                "Y2": y2
            })

    # Write annotated frame
    out.write(annotated_frame)

    # Progress
    if frame_number % 50 == 0:
        print(
            f"Processed {frame_number}/{total_frames} frames"
        )


# Release resources
cap.release()
out.release()

print("\nObject detection completed!")


# ============================================================
# 8. CREATE DETECTION DATAFRAME
# ============================================================

if len(detection_records) > 0:

    detections_df = pd.DataFrame(detection_records)

else:

    detections_df = pd.DataFrame(
        columns=[
            "Frame",
            "Object",
            "Confidence",
            "X1",
            "Y1",
            "X2",
            "Y2"
        ]
    )


print("\n================ DETECTION RESULTS ================")

print("Total detections:", len(detections_df))

display(detections_df.head(20))


# ============================================================
# 9. OBJECT DETECTION SUMMARY
# ============================================================

if len(detections_df) > 0:

    object_summary = (
        detections_df
        .groupby("Object")
        .agg(
            Detection_Count=("Object", "count"),
            Average_Confidence=("Confidence", "mean")
        )
        .reset_index()
        .sort_values(
            "Detection_Count",
            ascending=False
        )
    )

else:

    object_summary = pd.DataFrame(
        columns=[
            "Object",
            "Detection_Count",
            "Average_Confidence"
        ]
    )


print("\n================ OBJECT SUMMARY ================")

display(object_summary.round(3))


# ============================================================
# 10. GRAPH 1 — OBJECT DETECTION COUNTS
# ============================================================

plt.figure(figsize=(10, 6))

if len(object_summary) > 0:

    plt.bar(
        object_summary["Object"],
        object_summary["Detection_Count"]
    )

    plt.xlabel("Object Class")
    plt.ylabel("Number of Detections")
    plt.title("Objects Detected in CCTV Footage")
    plt.xticks(rotation=45)
    plt.tight_layout()

else:

    plt.text(
        0.5,
        0.5,
        "No objects detected",
        ha="center",
        va="center"
    )

    plt.axis("off")


graph1 = "01_object_detection_counts.png"

plt.savefig(
    graph1,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 11. GRAPH 2 — AVERAGE CONFIDENCE
# ============================================================

plt.figure(figsize=(10, 6))

if len(object_summary) > 0:

    plt.bar(
        object_summary["Object"],
        object_summary["Average_Confidence"]
    )

    plt.xlabel("Object Class")
    plt.ylabel("Average Confidence")
    plt.title("Average Detection Confidence")
    plt.xticks(rotation=45)
    plt.ylim(0, 1)
    plt.tight_layout()

else:

    plt.text(
        0.5,
        0.5,
        "No confidence data available",
        ha="center",
        va="center"
    )

    plt.axis("off")


graph2 = "02_average_confidence.png"

plt.savefig(
    graph2,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 12. GRAPH 3 — CONFIDENCE DISTRIBUTION
# ============================================================

plt.figure(figsize=(9, 6))

if len(detections_df) > 0:

    plt.hist(
        detections_df["Confidence"],
        bins=10
    )

    plt.xlabel("Confidence Score")
    plt.ylabel("Number of Detections")
    plt.title("Distribution of Detection Confidence")
    plt.tight_layout()

else:

    plt.text(
        0.5,
        0.5,
        "No confidence data available",
        ha="center",
        va="center"
    )

    plt.axis("off")


graph3 = "03_confidence_distribution.png"

plt.savefig(
    graph3,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 13. GRAPH 4 — DETECTIONS OVER VIDEO FRAMES
# ============================================================

plt.figure(figsize=(12, 6))

if len(detections_df) > 0:

    detections_per_frame = (
        detections_df
        .groupby("Frame")
        .size()
    )

    plt.plot(
        detections_per_frame.index,
        detections_per_frame.values
    )

    plt.xlabel("Frame Number")
    plt.ylabel("Number of Objects")
    plt.title("Object Detections Across Video Frames")
    plt.tight_layout()

else:

    plt.text(
        0.5,
        0.5,
        "No detections available",
        ha="center",
        va="center"
    )

    plt.axis("off")


graph4 = "04_detections_over_frames.png"

plt.savefig(
    graph4,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 14. DISPLAY OUTPUT VIDEO
# ============================================================

print("\n================ OUTPUT VIDEO ================")

display(
    Video(
        output_video,
        embed=True
    )
)


# ============================================================
# 15. SAVE CSV FILES
# ============================================================

detections_df.to_csv(
    "detection_results.csv",
    index=False
)

object_summary.to_csv(
    "object_summary.csv",
    index=False
)


# ============================================================
# 16. CALCULATE OBSERVATIONS
# ============================================================

if len(object_summary) > 0:

    most_detected_object = object_summary.iloc[0]["Object"]

    highest_detection_count = int(
        object_summary.iloc[0]["Detection_Count"]
    )

    average_confidence = detections_df["Confidence"].mean()

else:

    most_detected_object = "None"

    highest_detection_count = 0

    average_confidence = 0


observations = f"""
AIML ASSIGNMENT 9
OBJECT DETECTION FROM CCTV FOOTAGE

Model Used:
YOLOv8 Nano (yolov8n.pt)

Video:
{video_file}

Video Width:
{width}

Video Height:
{height}

Video FPS:
{fps:.2f}

Total Frames:
{total_frames}

Video Duration:
{duration:.2f} seconds

Confidence Threshold:
{CONFIDENCE_THRESHOLD}

Total Objects Detected:
{len(detections_df)}

Different Object Classes Detected:
{len(object_summary)}

Most Frequently Detected Object:
{most_detected_object}

Number of Detections of Most Frequent Object:
{highest_detection_count}

Average Detection Confidence:
{average_confidence:.4f}

MAIN OBSERVATIONS:

1. YOLO was used to perform real-time object detection on CCTV footage.

2. The model identifies objects using bounding boxes and class labels.

3. Each detected object is assigned a confidence score.

4. The confidence threshold was set to {CONFIDENCE_THRESHOLD}.

5. Multiple object classes can be detected within the same frame.

6. The output video contains bounding boxes around detected objects.

7. The object summary shows the frequency of detected object classes.

8. Higher confidence scores indicate that the model is more confident
   about its prediction.

9. Object detection can be applied to CCTV surveillance, traffic
   monitoring, security systems, crowd monitoring and smart-city
   applications.

10. Detection accuracy depends on video quality, lighting conditions,
    camera angle, object size and the pretrained model.
"""


print("\n================ OBSERVATIONS ================")
print(observations)


# ============================================================
# 17. CREATE HTML REPORT
# ============================================================

html_report = f"""
<!DOCTYPE html>

<html>

<head>

<meta charset="UTF-8">

<title>AIML Assignment 9 - Object Detection</title>

<style>

body {{
    font-family: Arial, sans-serif;
    margin: 40px;
    line-height: 1.6;
}}

h1, h2 {{
    color: #222;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-bottom: 25px;
}}

th, td {{
    border: 1px solid #999;
    padding: 8px;
    text-align: center;
}}

th {{
    background-color: #eeeeee;
}}

img {{
    max-width: 90%;
    margin: 20px 0;
}}

.observation {{
    background-color: #f4f4f4;
    padding: 15px;
    border-left: 5px solid #555;
}}

</style>

</head>

<body>

<h1>AIML Assignment 9</h1>

<h2>Apply Object Detection from CCTV Footage</h2>

<h2>Objective</h2>

<p>
The objective of this experiment is to detect and identify objects
present in CCTV footage using a pretrained YOLO object detection model.
The detected objects are represented using bounding boxes, class labels
and confidence scores.
</p>

<h2>Model Used</h2>

<p>
YOLOv8 Nano (yolov8n.pt)
</p>

<h2>Video Information</h2>

<table>

<tr>
<th>Property</th>
<th>Value</th>
</tr>

<tr>
<td>Video</td>
<td>{video_file}</td>
</tr>

<tr>
<td>Width</td>
<td>{width}</td>
</tr>

<tr>
<td>Height</td>
<td>{height}</td>
</tr>

<tr>
<td>FPS</td>
<td>{fps:.2f}</td>
</tr>

<tr>
<td>Total Frames</td>
<td>{total_frames}</td>
</tr>

<tr>
<td>Duration</td>
<td>{duration:.2f} seconds</td>
</tr>

<tr>
<td>Confidence Threshold</td>
<td>{CONFIDENCE_THRESHOLD}</td>
</tr>

</table>

<h2>Object Detection Summary</h2>

{object_summary.round(3).to_html(index=False)}

<h2>Object Detection Counts</h2>

<img src="01_object_detection_counts.png">

<h2>Average Confidence</h2>

<img src="02_average_confidence.png">

<h2>Confidence Distribution</h2>

<img src="03_confidence_distribution.png">

<h2>Detections Across Video Frames</h2>

<img src="04_detections_over_frames.png">

<h2>Observations</h2>

<div class="observation">

<pre>{observations}</pre>

</div>

<h2>Conclusion</h2>

<p>
The experiment successfully demonstrates object detection from CCTV
footage using YOLO. The model identifies objects and draws bounding
boxes around them. Object detection provides useful information for
automated surveillance, traffic monitoring, security systems and
other computer vision applications.
</p>

</body>

</html>
"""


with open(
    "05_Object_Detection_Complete_Report.html",
    "w",
    encoding="utf-8"
) as f:

    f.write(html_report)


# ============================================================
# 18. CREATE RESULTS FOLDER
# ============================================================

results_folder = "AIML_9_CCTV_Object_Detection_Results"

if os.path.exists(results_folder):

    shutil.rmtree(results_folder)

os.makedirs(results_folder)


# ============================================================
# 19. COPY ALL RESULTS
# ============================================================

files_to_copy = [

    output_video,

    graph1,
    graph2,
    graph3,
    graph4,

    "detection_results.csv",
    "object_summary.csv",

    "05_Object_Detection_Complete_Report.html"
]


for file in files_to_copy:

    if os.path.exists(file):

        shutil.copy(
            file,
            results_folder
        )


# ============================================================
# 20. SAVE OBSERVATIONS
# ============================================================

with open(
    os.path.join(
        results_folder,
        "06_results_and_observations.txt"
    ),
    "w",
    encoding="utf-8"
) as f:

    f.write(observations)


# ============================================================
# 21. CREATE ZIP
# ============================================================

zip_name = "AIML_9_CCTV_Object_Detection_Complete_Results"

shutil.make_archive(
    zip_name,
    "zip",
    results_folder
)


# ============================================================
# 22. FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 60)
print("ASSIGNMENT 9 COMPLETED SUCCESSFULLY")
print("=" * 60)

print("\nResults folder:")
print(results_folder)

print("\nGenerated files:")

for file in os.listdir(results_folder):

    print("✓", file)

print("\nZIP file:")
print(zip_name + ".zip")


# ============================================================
# 23. DOWNLOAD ZIP
# ============================================================

files.download(
    zip_name + ".zip"
)